In [1]:
import pandas as pd

In [2]:
file_path = "../data/raw/ibm_aml/HI-Small_Trans.csv"

sample = pd.read_csv(file_path, nrows=5)

sample

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [3]:
sample.columns.tolist()

['Timestamp',
 'From Bank',
 'Account',
 'To Bank',
 'Account.1',
 'Amount Received',
 'Receiving Currency',
 'Amount Paid',
 'Payment Currency',
 'Payment Format',
 'Is Laundering']

In [4]:
file_path = "../data/raw/ibm_aml/HI-Small_Trans.csv"

with open(file_path, "r", encoding="utf-8") as f:
    row_count = sum(1 for _ in f) - 1

print(f"Number of transactions: {row_count:,}")

Number of transactions: 5,078,345


In [5]:
labels = pd.read_csv(
    file_path,
    usecols=["Is Laundering"]
)

labels["Is Laundering"].value_counts()

Is Laundering
0    5073168
1       5177
Name: count, dtype: int64

In [6]:
laundering_count = labels["Is Laundering"].sum()
total_count = len(labels)

laundering_rate = laundering_count / total_count * 100

print(f"Total transactions: {total_count:,}")
print(f"Laundering transactions: {laundering_count:,}")
print(f"Laundering rate: {laundering_rate:.4f}%")

Total transactions: 5,078,345
Laundering transactions: 5,177
Laundering rate: 0.1019%


In [7]:
eda_sample = pd.read_csv(
    file_path,
    nrows=100000
)

eda_sample.isnull().sum()

Timestamp             0
From Bank             0
Account               0
To Bank               0
Account.1             0
Amount Received       0
Receiving Currency    0
Amount Paid           0
Payment Currency      0
Payment Format        0
Is Laundering         0
dtype: int64

In [8]:
eda_sample.dtypes

Timestamp              object
From Bank               int64
Account                object
To Bank                 int64
Account.1              object
Amount Received       float64
Receiving Currency     object
Amount Paid           float64
Payment Currency       object
Payment Format         object
Is Laundering           int64
dtype: object

In [9]:
print("Payment Formats:")
print(eda_sample["Payment Format"].value_counts())

print("\nPayment Currencies:")
print(eda_sample["Payment Currency"].value_counts())

print("\nReceiving Currencies:")
print(eda_sample["Receiving Currency"].value_counts())

Payment Formats:
Payment Format
Reinvestment    73296
Cheque           9393
Credit Card      9028
ACH              4436
Cash             2668
Wire             1149
Bitcoin            30
Name: count, dtype: int64

Payment Currencies:
Payment Currency
US Dollar            99627
Euro                   126
Yuan                    86
Canadian Dollar         32
Rupee                   29
Bitcoin                 29
UK Pound                23
Yen                     21
Australian Dollar        7
Mexican Peso             7
Ruble                    7
Swiss Franc              6
Name: count, dtype: int64

Receiving Currencies:
Receiving Currency
US Dollar            99816
Euro                    71
Bitcoin                 30
UK Pound                14
Rupee                   13
Yuan                    13
Canadian Dollar          9
Yen                      8
Ruble                    7
Mexican Peso             7
Australian Dollar        6
Swiss Franc              6
Name: count, dtype: int64


In [10]:
eda_sample["Is Laundering"].value_counts()

Is Laundering
0    99995
1        5
Name: count, dtype: int64

In [11]:
eda_sample = eda_sample.rename(columns={
    "Account": "From Account",
    "Account.1": "To Account"
})

eda_sample.columns.tolist()

['Timestamp',
 'From Bank',
 'From Account',
 'To Bank',
 'To Account',
 'Amount Received',
 'Receiving Currency',
 'Amount Paid',
 'Payment Currency',
 'Payment Format',
 'Is Laundering']

In [12]:
eda_sample["Timestamp"] = pd.to_datetime(
    eda_sample["Timestamp"]
)

eda_sample["Timestamp"].dtype

dtype('<M8[ns]')

In [13]:
print("Earliest transaction:", eda_sample["Timestamp"].min())
print("Latest transaction:", eda_sample["Timestamp"].max())

Earliest transaction: 2022-09-01 00:00:00
Latest transaction: 2022-09-01 00:29:00


In [14]:
print("Unique sending banks:", eda_sample["From Bank"].nunique())
print("Unique receiving banks:", eda_sample["To Bank"].nunique())

print("Unique sending accounts:", eda_sample["From Account"].nunique())
print("Unique receiving accounts:", eda_sample["To Account"].nunique())

Unique sending banks: 4504
Unique receiving banks: 3808
Unique sending accounts: 74761
Unique receiving accounts: 73022


In [15]:
eda_sample[["Amount Received", "Amount Paid"]].describe()

,Amount Received,Amount Paid
count,1.000000e+05,1.000000e+05
mean,7.555877e+05,7.563408e+05
std,2.282190e+07,2.282226e+07
min,3.327000e-03,3.327000e-03
25%,2.044000e+01,2.045000e+01
50%,2.090525e+03,2.101495e+03
75%,2.391585e+04,2.395517e+04
max,5.351189e+09,5.351189e+09


In [16]:
amount_diff = eda_sample[
    eda_sample["Amount Paid"] != eda_sample["Amount Received"]
]

print("Transactions with different paid/received amounts:", len(amount_diff))
print("Percentage:", len(amount_diff) / len(eda_sample) * 100)

Transactions with different paid/received amounts: 288
Percentage: 0.28800000000000003


In [17]:
currency_diff = eda_sample[
    eda_sample["Payment Currency"] != eda_sample["Receiving Currency"]
]

print("Transactions with different currencies:", len(currency_diff))
print("Percentage:", len(currency_diff) / len(eda_sample) * 100)

Transactions with different currencies: 289
Percentage: 0.28900000000000003


In [18]:
amount_changed = eda_sample["Amount Paid"] != eda_sample["Amount Received"]
currency_changed = eda_sample["Payment Currency"] != eda_sample["Receiving Currency"]

print(pd.crosstab(
    currency_changed,
    amount_changed,
    rownames=["Currency Changed"],
    colnames=["Amount Changed"]
))

Amount Changed    False  True 
Currency Changed              
False             99711      0
True                  1    288


In [19]:
same_amount_different_currency = eda_sample[
    currency_changed & ~amount_changed
]

same_amount_different_currency

,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
55604,2022-09-01 00:24:00,322191,8086F8260,322191,8086F8260,0.01,US Dollar,0.01,Canadian Dollar,ACH,0


In [20]:
duplicate_count = eda_sample.duplicated().sum()

print("Duplicate transactions:", duplicate_count)
print("Duplicate percentage:", duplicate_count / len(eda_sample) * 100)

Duplicate transactions: 0
Duplicate percentage: 0.0


In [21]:
laundering_chunks = []

for chunk in pd.read_csv(file_path, chunksize=100000):
    laundering_rows = chunk[chunk["Is Laundering"] == 1]
    laundering_chunks.append(laundering_rows)

all_laundering = pd.concat(laundering_chunks, ignore_index=True)

print("Laundering transactions found:", len(all_laundering))

Laundering transactions found: 5177


In [22]:
legitimate_chunks = []

for chunk in pd.read_csv(file_path, chunksize=100000):
    legitimate_rows = chunk[chunk["Is Laundering"] == 0]
    
    sample = legitimate_rows.sample(
        frac=0.0102,
        random_state=42
    )
    
    legitimate_chunks.append(sample)

sampled_legitimate = pd.concat(
    legitimate_chunks,
    ignore_index=True
)

print("Legitimate transactions sampled:", len(sampled_legitimate))

Legitimate transactions sampled: 51747


In [23]:
model_data = pd.concat(
    [sampled_legitimate, all_laundering],
    ignore_index=True
)

print("Total modelling transactions:", len(model_data))
print()
print(model_data["Is Laundering"].value_counts())

Total modelling transactions: 56924

Is Laundering
0    51747
1     5177
Name: count, dtype: int64


In [24]:
model_data = model_data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

model_data["Is Laundering"].head(20).tolist()

[0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [25]:
model_data = model_data.rename(columns={
    "Account": "From Account",
    "Account.1": "To Account"
})

model_data.columns.tolist()

['Timestamp',
 'From Bank',
 'From Account',
 'To Bank',
 'To Account',
 'Amount Received',
 'Receiving Currency',
 'Amount Paid',
 'Payment Currency',
 'Payment Format',
 'Is Laundering']

In [26]:
model_data["Timestamp"] = pd.to_datetime(
    model_data["Timestamp"]
)

print(model_data["Timestamp"].dtype)

datetime64[ns]


In [27]:
output_path = "../data/processed/ibm_aml_model_data.csv"

model_data.to_csv(
    output_path,
    index=False
)

print("Saved modelling dataset to:", output_path)
print("Rows saved:", len(model_data))

Saved modelling dataset to: ../data/processed/ibm_aml_model_data.csv
Rows saved: 56924


In [28]:
check_data = pd.read_csv("../data/processed/ibm_aml_model_data.csv")

print("Shape:", check_data.shape)
print()
print(check_data["Is Laundering"].value_counts())

Shape: (56924, 11)

Is Laundering
0    51747
1     5177
Name: count, dtype: int64


In [29]:
accounts_path = "../data/raw/ibm_aml/HI-Small_accounts.csv"

accounts_sample = pd.read_csv(
    accounts_path,
    nrows=5
)

accounts_sample

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889


In [30]:
accounts = pd.read_csv(accounts_path)

print("Accounts dataset shape:", accounts.shape)

Accounts dataset shape: (518581, 5)


In [31]:
duplicate_accounts = accounts.duplicated(
    subset=["Bank ID", "Account Number"]
).sum()

print("Duplicate bank-account combinations:", duplicate_accounts)

Duplicate bank-account combinations: 0


In [32]:
accounts.isnull().sum()

Bank Name         0
Bank ID           0
Account Number    0
Entity ID         0
Entity Name       0
dtype: int64

In [33]:
from_accounts = model_data[
    ["From Bank", "From Account"]
].drop_duplicates()

matched_from = from_accounts.merge(
    accounts,
    left_on=["From Bank", "From Account"],
    right_on=["Bank ID", "Account Number"],
    how="left",
    indicator=True
)

print(matched_from["_merge"].value_counts())

_merge
both          41574
left_only         0
right_only        0
Name: count, dtype: int64


In [34]:
to_accounts = model_data[
    ["To Bank", "To Account"]
].drop_duplicates()

matched_to = to_accounts.merge(
    accounts,
    left_on=["To Bank", "To Account"],
    right_on=["Bank ID", "Account Number"],
    how="left",
    indicator=True
)

print(matched_to["_merge"].value_counts())

_merge
both          49080
left_only         0
right_only        0
Name: count, dtype: int64


In [35]:
print("Unique entities:", accounts["Entity ID"].nunique())
print("Unique entity names:", accounts["Entity Name"].nunique())
print("Unique banks:", accounts["Bank ID"].nunique())

Unique entities: 166207
Unique entity names: 166207
Unique banks: 30470


In [36]:
accounts_per_entity = accounts.groupby("Entity ID")["Account Number"].nunique()

print("Average accounts per entity:", accounts_per_entity.mean())
print("Maximum accounts for one entity:", accounts_per_entity.max())
print("Entities with multiple accounts:", (accounts_per_entity > 1).sum())

Average accounts per entity: 3.1200912115614865
Maximum accounts for one entity: 7820
Entities with multiple accounts: 63518


In [37]:
features = model_data.copy()

print("Feature dataset shape:", features.shape)

Feature dataset shape: (56924, 11)


In [38]:
features["Transaction Hour"] = features["Timestamp"].dt.hour
features["Day of Week"] = features["Timestamp"].dt.dayofweek
features["Is Weekend"] = (
    features["Timestamp"].dt.dayofweek >= 5
).astype(int)

features[
    ["Timestamp", "Transaction Hour", "Day of Week", "Is Weekend"]
].head()

,Timestamp,Transaction Hour,Day of Week,Is Weekend
0,2022-09-04 10:31:00,10,6,1
1,2022-09-07 06:37:00,6,2,0
2,2022-09-05 13:14:00,13,0,0
3,2022-09-02 01:36:00,1,4,0
4,2022-09-02 07:55:00,7,4,0


In [39]:
features["Is Cross Currency"] = (
    features["Payment Currency"] != features["Receiving Currency"]
).astype(int)

print(features["Is Cross Currency"].value_counts())

Is Cross Currency
0    56157
1      767
Name: count, dtype: int64


In [40]:
features["Is Same Bank"] = (
    features["From Bank"] == features["To Bank"]
).astype(int)

print(features["Is Same Bank"].value_counts())

Is Same Bank
0    49677
1     7247
Name: count, dtype: int64


In [41]:
features["Is Same Account"] = (
    (features["From Bank"] == features["To Bank"]) &
    (features["From Account"] == features["To Account"])
).astype(int)

print(features["Is Same Account"].value_counts())

Is Same Account
0    50804
1     6120
Name: count, dtype: int64


In [43]:
import numpy as np

features["Log Amount Paid"] = np.log1p(
    features["Amount Paid"]
)

features[
    ["Amount Paid", "Log Amount Paid"]
].head()

,Amount Paid,Log Amount Paid
0,1917152.50,14.466352
1,59.05,4.095178
2,7834.10,8.966369
3,3403.41,8.132827
4,67766.83,11.123843


In [44]:
features["Amount Difference"] = abs(
    features["Amount Paid"] - features["Amount Received"]
)

print(features["Amount Difference"].describe())

count    5.692400e+04
mean     4.271546e+05
std      4.445114e+07
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      7.059147e+09
Name: Amount Difference, dtype: float64


In [45]:
sender_lookup = accounts[
    ["Bank ID", "Account Number", "Entity ID"]
].rename(columns={
    "Bank ID": "From Bank",
    "Account Number": "From Account",
    "Entity ID": "From Entity ID"
})

features = features.merge(
    sender_lookup,
    on=["From Bank", "From Account"],
    how="left"
)

print("Missing sender entities:", features["From Entity ID"].isnull().sum())
print("Feature dataset shape:", features.shape)

Missing sender entities: 0
Feature dataset shape: (56924, 20)


In [46]:
features.columns.tolist()

['Timestamp',
 'From Bank',
 'From Account',
 'To Bank',
 'To Account',
 'Amount Received',
 'Receiving Currency',
 'Amount Paid',
 'Payment Currency',
 'Payment Format',
 'Is Laundering',
 'Transaction Hour',
 'Day of Week',
 'Is Weekend',
 'Is Cross Currency',
 'Is Same Bank',
 'Is Same Account',
 'Log Amount Paid',
 'Amount Difference',
 'From Entity ID']

In [47]:
receiver_lookup = accounts[
    ["Bank ID", "Account Number", "Entity ID"]
].rename(columns={
    "Bank ID": "To Bank",
    "Account Number": "To Account",
    "Entity ID": "To Entity ID"
})

features = features.merge(
    receiver_lookup,
    on=["To Bank", "To Account"],
    how="left"
)

print("Missing receiver entities:", features["To Entity ID"].isnull().sum())
print("Feature dataset shape:", features.shape)

Missing receiver entities: 0
Feature dataset shape: (56924, 21)


In [48]:
features["Is Same Entity"] = (
    features["From Entity ID"] == features["To Entity ID"]
).astype(int)

print(features["Is Same Entity"].value_counts())

Is Same Entity
0    50022
1     6902
Name: count, dtype: int64


In [49]:
features["Entity Pair"] = (
    features["From Entity ID"].astype(str)
    + " -> "
    + features["To Entity ID"].astype(str)
)

print("Unique entity pairs:", features["Entity Pair"].nunique())

features[
    ["From Entity ID", "To Entity ID", "Entity Pair"]
].head()

Unique entity pairs: 50107


,From Entity ID,To Entity ID,Entity Pair
0,8001C6440,800183B60,8001C6440 -> 800183B60
1,800CD3AC0,8006FBAD0,800CD3AC0 -> 8006FBAD0
2,8000A1BC0,80037B340,8000A1BC0 -> 80037B340
3,800227E00,800F1BD10,800227E00 -> 800F1BD10
4,800AC8F90,800AAEB80,800AC8F90 -> 800AAEB80


In [50]:
features["Entity Pair Transaction Count"] = (
    features.groupby("Entity Pair")["Entity Pair"].transform("count")
)

print(features["Entity Pair Transaction Count"].describe())

count    56924.000000
mean         8.938409
std         59.727236
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max        583.000000
Name: Entity Pair Transaction Count, dtype: float64


In [51]:
features["Sender Transaction Count"] = (
    features.groupby("From Entity ID")["From Entity ID"].transform("count")
)

print(features["Sender Transaction Count"].describe())

count    56924.000000
mean       129.820252
std        409.985357
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max       1962.000000
Name: Sender Transaction Count, dtype: float64


In [52]:
features["Receiver Transaction Count"] = (
    features.groupby("To Entity ID")["To Entity ID"].transform("count")
)

print(features["Receiver Transaction Count"].describe())

count    56924.000000
mean        23.049469
std         98.722779
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        755.000000
Name: Receiver Transaction Count, dtype: float64


In [53]:
features["Log Amount Difference"] = np.log1p(
    features["Amount Difference"]
)

features[
    ["Amount Difference", "Log Amount Difference"]
].head()

,Amount Difference,Log Amount Difference
0,0.0,0.0
1,0.0,0.0
2,0.0,0.0
3,0.0,0.0
4,0.0,0.0


In [54]:
print("Dataset shape:", features.shape)
print()
print(features.columns.tolist())

Dataset shape: (56924, 27)

['Timestamp', 'From Bank', 'From Account', 'To Bank', 'To Account', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering', 'Transaction Hour', 'Day of Week', 'Is Weekend', 'Is Cross Currency', 'Is Same Bank', 'Is Same Account', 'Log Amount Paid', 'Amount Difference', 'From Entity ID', 'To Entity ID', 'Is Same Entity', 'Entity Pair', 'Entity Pair Transaction Count', 'Sender Transaction Count', 'Receiver Transaction Count', 'Log Amount Difference']


In [55]:
binary_features = [
    "Is Weekend",
    "Is Cross Currency",
    "Is Same Bank",
    "Is Same Account",
    "Is Same Entity"
]

feature_comparison = features.groupby("Is Laundering")[binary_features].mean()

feature_comparison

,Is Weekend,Is Cross Currency,Is Same Bank,Is Same Account,Is Same Entity
Is Laundering,,,,,
0,0.123041,0.014822,0.138056,0.118055,0.132703
1,0.288777,0.000000,0.019896,0.002125,0.006761


In [56]:
numeric_features = [
    "Log Amount Paid",
    "Log Amount Difference",
    "Entity Pair Transaction Count",
    "Sender Transaction Count",
    "Receiver Transaction Count"
]

features.groupby("Is Laundering")[numeric_features].median()


,Log Amount Paid,Log Amount Difference,Entity Pair Transaction Count,Sender Transaction Count,Receiver Transaction Count
Is Laundering,,,,,
0,7.248127,0.0,1.0,2.0,1.0
1,9.067418,0.0,1.0,2.0,2.0


In [57]:
engineered_features = [
    "Transaction Hour",
    "Day of Week",
    "Is Weekend",
    "Is Cross Currency",
    "Is Same Bank",
    "Is Same Account",
    "Log Amount Paid",
    "Amount Difference",
    "Is Same Entity",
    "Entity Pair Transaction Count",
    "Sender Transaction Count",
    "Receiver Transaction Count",
    "Log Amount Difference"
]

print("Missing values:")
print(features[engineered_features].isnull().sum())

print()
print(
    "Infinite values:",
    np.isinf(features[engineered_features]).sum().sum()
)

Missing values:
Transaction Hour                 0
Day of Week                      0
Is Weekend                       0
Is Cross Currency                0
Is Same Bank                     0
Is Same Account                  0
Log Amount Paid                  0
Amount Difference                0
Is Same Entity                   0
Entity Pair Transaction Count    0
Sender Transaction Count         0
Receiver Transaction Count       0
Log Amount Difference            0
dtype: int64

Infinite values: 0


In [58]:
y = features["Is Laundering"].copy()

print("Total labels:", len(y))
print()
print(y.value_counts())

Total labels: 56924

Is Laundering
0    51747
1     5177
Name: count, dtype: int64


In [59]:
model_feature_columns = [
    "Log Amount Paid",
    "Transaction Hour",
    "Day of Week",
    "Is Weekend",
    "Is Cross Currency",
    "Is Same Bank",
    "Is Same Account",
    "Is Same Entity",
    "Payment Format"
]

X = features[model_feature_columns].copy()

print("X shape:", X.shape)
print()
print(X.columns.tolist())

X shape: (56924, 9)

['Log Amount Paid', 'Transaction Hour', 'Day of Week', 'Is Weekend', 'Is Cross Currency', 'Is Same Bank', 'Is Same Account', 'Is Same Entity', 'Payment Format']


In [60]:
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: divide temporary data equally
# → 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (39846, 9) (39846,)
Validation: (8539, 9) (8539,)
Test: (8539, 9) (8539,)


In [61]:
print("Training laundering rate:")
print(y_train.value_counts(normalize=True))

print("\nValidation laundering rate:")
print(y_val.value_counts(normalize=True))

print("\nTest laundering rate:")
print(y_test.value_counts(normalize=True))

Training laundering rate:
Is Laundering
0    0.90905
1    0.09095
Name: proportion, dtype: float64

Validation laundering rate:
Is Laundering
0    0.909006
1    0.090994
Name: proportion, dtype: float64

Test laundering rate:
Is Laundering
0    0.909123
1    0.090877
Name: proportion, dtype: float64


In [62]:
print("TRAIN")
print(y_train.value_counts())

print("\nVALIDATION")
print(y_val.value_counts())

print("\nTEST")
print(y_test.value_counts())

TRAIN
Is Laundering
0    36222
1     3624
Name: count, dtype: int64

VALIDATION
Is Laundering
0    7762
1     777
Name: count, dtype: int64

TEST
Is Laundering
0    7763
1     776
Name: count, dtype: int64


In [63]:
numeric_features = [
    "Log Amount Paid",
    "Transaction Hour",
    "Day of Week",
    "Is Weekend",
    "Is Cross Currency",
    "Is Same Bank",
    "Is Same Account",
    "Is Same Entity"
]

categorical_features = [
    "Payment Format"
]

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total:", len(numeric_features) + len(categorical_features))

Numerical features: 8
Categorical features: 1
Total: 9


In [64]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

print(preprocessor)

ColumnTransformer(remainder='passthrough',
                  transformers=[('categorical',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['Payment Format'])])


In [65]:
preprocessor.fit(X_train)

print("Preprocessor fitted successfully.")

Preprocessor fitted successfully.


In [66]:
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (39846, 15)
Processed validation shape: (8539, 15)
Processed test shape: (8539, 15)


In [67]:
processed_feature_names = preprocessor.get_feature_names_out()

print(processed_feature_names)

['categorical__Payment Format_ACH' 'categorical__Payment Format_Bitcoin'
 'categorical__Payment Format_Cash' 'categorical__Payment Format_Cheque'
 'categorical__Payment Format_Credit Card'
 'categorical__Payment Format_Reinvestment'
 'categorical__Payment Format_Wire' 'remainder__Log Amount Paid'
 'remainder__Transaction Hour' 'remainder__Day of Week'
 'remainder__Is Weekend' 'remainder__Is Cross Currency'
 'remainder__Is Same Bank' 'remainder__Is Same Account'
 'remainder__Is Same Entity']


In [68]:
target_leaked = any(
    "Is Laundering" in feature
    for feature in processed_feature_names
)

print("Target present in model features:", target_leaked)

Target present in model features: False


In [69]:
print("Training missing values:", np.isnan(X_train_processed).sum())
print("Validation missing values:", np.isnan(X_val_processed).sum())
print("Test missing values:", np.isnan(X_test_processed).sum())

Training missing values: 0
Validation missing values: 0
Test missing values: 0


In [70]:
print("FINAL PHASE 2 DATA SUMMARY")
print("--------------------------")
print("Training X:", X_train_processed.shape)
print("Training y:", y_train.shape)

print("\nValidation X:", X_val_processed.shape)
print("Validation y:", y_val.shape)

print("\nTest X:", X_test_processed.shape)
print("Test y:", y_test.shape)

print("\nNumber of final features:", len(processed_feature_names))

FINAL PHASE 2 DATA SUMMARY
--------------------------
Training X: (39846, 15)
Training y: (39846,)

Validation X: (8539, 15)
Validation y: (8539,)

Test X: (8539, 15)
Test y: (8539,)

Number of final features: 15
